# From an optimal map to an inverse gap

This notebook reconstructs the two continuous models in Figures 2–4. It runs quickly at small sample sizes. The exact article settings and cached large runs are explored in notebook 03.

The cost is $c_A(x,y)=-x^\top Ay$, with $A(u)=\operatorname{diag}(u,1-u)$ and $0\leq u\leq1$. For observed pairs $Y_i=T(X_i)$ the empirical mean gap is
$$L_n(u)=\max_{\sigma\in\mathfrak S_n}\frac1n\sum_i X_i^\top A(u)Y_{\sigma(i)}-\frac1n\sum_i X_i^\top A(u)Y_i.$$
Every value below is computed by exact assignment, without entropy. Straight line segments join evaluated points; they do not assert that all intervening assignment breakpoints have been found.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "iot_gap").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
from iot_gap import (cosine_map, annulus_map, sample_pairs, sample_independent_marginals,
                     optimal_assignment, gap_curve, sampling_gap_curve)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
US = np.linspace(0, 1, 51)

## A convex gradient perturbation
$$\Psi_\delta(x)=\frac12|x|^2+\delta[\cos x_1+\cos x_2+\cos(x_1+x_2)],\qquad T_\delta=\nabla\Psi_\delta.$$
For $0\leq\delta<1/3$, $\nabla^2\Psi_\delta\succeq(1-3\delta)I$. Thus $Y=T_\delta(X)$ with $X\sim N(0,I_2)$ samples the true OT plan. Positive $\delta$ identifies the population matrix up to scale, whereas $\delta=0$ leaves the whole PSD slice compatible. Uniform convexity is lost at the boundary $\delta=1/3$.

In [ ]:
def cosine_hessian(X, delta):
    H = np.tile(np.eye(2), (len(X), 1, 1))
    common = np.cos(X.sum(axis=1))
    H[:, 0, 0] -= delta * (np.cos(X[:, 0]) + common)
    H[:, 1, 1] -= delta * (np.cos(X[:, 1]) + common)
    H[:, 0, 1] = H[:, 1, 0] = -delta * common
    return H

X, Y = sample_pairs("cosine", 100, seed=100, delta=.32)
minimum_eigenvalue = np.linalg.eigvalsh(cosine_hessian(X, .32)).min()
assert minimum_eigenvalue >= 1 - 3*.32 - 1e-12
print("Sample minimum Hessian eigenvalue:", minimum_eigenvalue)
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(*X.T, s=12, label="X: standard Gaussian")
ax.scatter(*Y.T, s=12, label="Y = Tδ(X)")
ax.quiver(*X.T, *(Y-X).T, angles="xy", scale_units="xy", scale=1, alpha=.3)
ax.set_aspect("equal"); ax.legend(); ax.set_title("Observed true-plan pairs, δ = 0.32")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for delta in [0, .01, .1, .3, .329]:
    X, Y = sample_pairs("cosine", 100, seed=100, delta=delta)
    loss = gap_curve(X, Y, US)
    assert loss.min() >= 0 and abs(loss[len(US)//2]) < 1e-12
    axes[0].plot(US, loss, label=f"δ={delta:g}")
    if delta > 0:
        axes[1].plot(US, loss/delta**2, label=f"δ={delta:g}")
    else:
        assert np.max(loss) < 1e-12
axes[0].set(title="Raw mean gap", xlabel="u", ylabel="Lₙ(u)")
axes[1].set(title="Same losses divided by δ²", xlabel="u", ylabel="Lₙ(u) / δ²")
for ax in axes: ax.legend(fontsize=8); ax.axvline(.5, color="grey", lw=.7, ls=":")
fig.tight_layout(); plt.show()

Dividing by $\delta^2$ changes the vertical scale. It cannot remove a finite-sample zero plateau. The normalized curves need not coincide: the sample assignment combinatorics and the perturbation size both matter. The case $\delta=0$ is deliberately omitted from this normalization.

## A radial map to an annulus
For $a=0.8$, $b=0.85$,
$$T(x)=\frac{x}{|x|}\sqrt{a^2+(b^2-a^2)(1-e^{-|x|^2/2})}.$$
The squared target radius is uniform on $[a^2,b^2]$. The map is defined almost everywhere and is discontinuous at the origin; the implementation chooses a measurable value at that null point. No $C^1$ claim at the origin is needed.

In [ ]:
X, Y = sample_pairs("annulus", 100, seed=100)
quantiles = (np.sum(Y**2, axis=1) - .8**2)/(.85**2-.8**2)
assert np.allclose(quantiles, -np.expm1(-np.sum(X**2, axis=1)/2))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].scatter(*X.T, s=12, label="Gaussian source")
axes[0].scatter(*Y.T, s=12, label="Paired annulus target")
axes[0].set_aspect("equal"); axes[0].legend(fontsize=8)
axes[1].plot(US, gap_curve(X, Y, US), color="#7851a9")
axes[1].set(xlabel="u", ylabel="Mean gap", title="Radial population identification; empirical plateau")
fig.tight_layout(); plt.show()

## Optional live controls
The next cell creates a widget in Jupyter. Change n, δ and the model, then press **Run experiment**. It uses small sample sizes for exploration; it does not overwrite article data.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    def explore(model="cosine", n=80, delta=.1, seed=10):
        X, Y = sample_pairs(model, n, seed, delta=delta)
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.plot(US, gap_curve(X, Y, US)); ax.set(xlabel="u", ylabel="Mean gap", title=f"{model}; n={n}; δ={delta:g}")
        plt.show()
    display(widgets.interact_manual(explore, model=["cosine", "annulus"], n=[30, 80, 160],
                                   delta=widgets.FloatSlider(value=.1, min=0, max=.329, step=.001), seed=(1, 50)))
except ImportError:
    print("Install the notebooks extra to enable widgets.")